# 작은 모델 파인튜닝 — Arduino 함수 호출 (Lab.3 Part E)

Part C의 함수(`set_led`, `set_servo`, `read_light`)에 맞춘 예문으로 작은 모델을 추가 학습한다. **같은 노트북으로 두 모델을 학습할 수 있다.**

| `MODEL_CHOICE` | 모델 | 학습 방법 |
|---|---|---|
| `"functiongemma"` | FunctionGemma 270M (함수 호출 전용) | 전체 학습 (단어 사전만 고정) |
| `"qwen3-0.6b"` | Qwen3 0.6B (범용) | LoRA (작은 보조 가중치만 학습한 뒤 합치기) |

**순서**: ① 준비 → ② 학습 데이터 생성 → ③ 학습 전 정확도 → ④ 학습 → ⑤ 학습 후 정확도·저장 → ⑥ GGUF 변환 → ⑦ 다운로드

**런타임**: `런타임 → 런타임 유형 변경 → T4 GPU` (무료). 학습 시간은 RTX 3070 Ti(노트북) 기준 FunctionGemma 약 5분, Qwen3 0.6B 약 25분 (예문이 더 길고 모델이 더 크다). T4는 bf16을 지원하지 않아 fp32로 학습하므로 더 걸릴 수 있다.

**두 모델을 모두 학습하려면**: 한 모델을 ⑦까지 끝낸 뒤 `런타임 → 세션 다시 시작` → 아래 `MODEL_CHOICE`를 바꾸고 → ①의 두 번째 셀부터 다시 실행한다.

## ① 준비

실습 파일 4개가 필요하다. 저장소를 clone하거나, 왼쪽 **파일** 패널에 직접 업로드한다.

`fc_b_tools_api.py`, `fc_c_arduino_agent.py`, `make_train_data.py`, `eval_set.jsonl`

In [ ]:
!pip install -q -U "transformers>=4.57" accelerate sentencepiece peft
import os
if not os.path.exists("fc_c_arduino_agent.py"):
    !git clone -q --depth 1 https://github.com/bgloh/ai_iot.git
    %cd ai_iot/FunctionCalling
!ls

In [ ]:
# ★ 학습할 모델을 고른다: "functiongemma" 또는 "qwen3-0.6b"
MODEL_CHOICE = "functiongemma"

CONFIGS = {
    "functiongemma": dict(model_id="unsloth/functiongemma-270m-it",   # google/functiongemma-270m-it 와 같은 가중치 (로그인 불필요)
                          method="full", lr=5e-5, out_dir="functiongemma-arduino"),
    "qwen3-0.6b":    dict(model_id="Qwen/Qwen3-0.6B",
                          method="lora", lr=2e-4, out_dir="qwen3-0.6b-arduino"),
}
cfg = CONFIGS[MODEL_CHOICE]
MODEL_ID, OUT_DIR = cfg["model_id"], cfg["out_dir"]

import json, re, time, random, shutil
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from fc_c_arduino_agent import TOOLS, SYSTEM_PROMPT

device = "cuda"
use_bf16 = torch.cuda.is_bf16_supported()     # T4는 False → fp32로 학습
print(MODEL_CHOICE, "|", torch.cuda.get_device_name(0), "| bf16:", use_bf16)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(device)
print(f"파라미터 {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

# Qwen3의 '생각하기' 단계를 끈다 (Jetson 서버의 --reasoning-budget 0 과 같은 조건). FunctionGemma는 이 값을 무시한다.
TEMPLATE_KW = dict(tools=TOOLS, enable_thinking=False)

## ② 학습 데이터 생성

`make_train_data.py`가 여러 말투의 명령과 정답 함수 호출을 만든다. 평가 문장(`eval_set.jsonl`)과 같은 문장은 자동으로 뺀다. 두 모델 모두 **같은 데이터**로 학습한다.

In [ ]:
!python make_train_data.py --n 1500
train_rows = [json.loads(l) for l in open("train.jsonl", encoding="utf-8")]
eval_rows = [json.loads(l) for l in open("eval_set.jsonl", encoding="utf-8")]
print(len(train_rows), "학습 /", len(eval_rows), "평가")
print(json.dumps(train_rows[0], ensure_ascii=False, indent=1)[:600])

## ③ 학습 전 정확도

Jetson의 `fc_eval.py`와 같은 기준으로 채점한다. 두 모델의 함수 호출 양식을 모두 읽는다.

- FunctionGemma: `<start_function_call>call:set_led{state:<escape>on<escape>}<end_function_call>`
- Qwen3: `<tool_call>{"name": "set_led", "arguments": {"state": "on"}}</tool_call>`

In [ ]:
FG_RE = re.compile(r"<start_function_call>call:(\w+)\{(.*?)\}<end_function_call>", re.S)
FG_ARG = re.compile(r"(\w+):(?:<escape>(.*?)<escape>|([^,}]*))", re.S)
QWEN_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)

def norm(v):
    if isinstance(v, str):
        v = v.strip()
        return int(v) if v.lstrip("-").isdigit() else v.lower()
    return v

def parse_call(text):
    m = FG_RE.search(text)
    if m:
        args = {k: norm(s if s else raw) for k, s, raw in FG_ARG.findall(m.group(2))}
        return m.group(1), args
    m = QWEN_RE.search(text)
    if m:
        try:
            d = json.loads(m.group(1))
            return d["name"], {k: norm(v) for k, v in (d.get("arguments") or {}).items()}
        except (ValueError, KeyError):
            return "INVALID", {}
    return None

def predict(text):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt",
                                  return_dict=True, **TEMPLATE_KW).to(device)
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
        out = model.generate(**ids, max_new_tokens=64, do_sample=False)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=False)

def evaluate(show=False):
    model.eval()
    ok = 0
    for row in eval_rows:
        got = parse_call(predict(row["text"]))
        exp = row["expect"]
        if not exp:
            good = got is None
        else:
            good = got is not None and got[0] == exp[0]["name"] and got[1] == exp[0]["arguments"]
        ok += good
        if show and not good:
            print("X", row["text"], "| 정답", exp[:1], "| 출력", got)
    print(f"정확도 {ok}/{len(eval_rows)} = {ok / len(eval_rows) * 100:.1f}%")
    return ok / len(eval_rows)

before = evaluate(show=True)

## ④ 학습

- 마지막 assistant 출력(함수 호출 또는 답변)에만 loss를 건다 (프롬프트 부분은 `-100`으로 가림).
- **FunctionGemma (전체 학습)**: 단어 사전(임베딩, 어휘 26만 개로 전체의 절반 이상)은 고정하고 나머지를 모두 학습한다.
- **Qwen3 0.6B (LoRA)**: 원래 가중치는 모두 고정하고, 각 층에 작은 보조 행렬(rank 16)을 붙여 그것만 학습한다. 전체 학습보다 메모리를 훨씬 적게 쓴다.

In [ ]:
def encode(messages):
    """[정답 직전까지 = 프롬프트] + [정답] 으로 나누고, 정답 부분만 loss를 계산하게 한다."""
    prev = messages[:-1]
    full = tok.apply_chat_template(messages, tokenize=False, **TEMPLATE_KW)
    for gen in (True, False):   # 모델마다 정답 앞에 '답변 시작 표시'가 붙는지가 다르다
        prefix = tok.apply_chat_template(prev, tokenize=False, add_generation_prompt=gen, **TEMPLATE_KW)
        if full.startswith(prefix):
            break
    else:
        raise ValueError("프롬프트와 정답을 나눌 수 없음")
    p = tok(prefix, add_special_tokens=False)["input_ids"]
    t = tok(full[len(prefix):], add_special_tokens=False)["input_ids"]
    return p + t, [-100] * len(p) + t

data = [encode(r["messages"]) for r in train_rows]
print("최대 길이", max(len(x) for x, _ in data), "토큰")
print("정답 부분 예:", repr(tok.decode([t for t in data[0][1] if t != -100])))

if cfg["method"] == "full":
    for p in model.get_input_embeddings().parameters():
        p.requires_grad = False
else:
    from peft import LoraConfig, get_peft_model
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                      "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, lora)
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"학습 파라미터 {sum(p.numel() for p in trainable) / 1e6:.1f}M")

EPOCHS, BATCH, ACCUM, LR = 2, 2, 8, cfg["lr"]   # 메모리가 부족하면 BATCH를 줄이고 ACCUM을 늘린다
opt = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.0)
steps = EPOCHS * len(data) // (BATCH * ACCUM)
sched = get_cosine_schedule_with_warmup(opt, int(steps * 0.05), steps)
pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

def batches():
    order = list(range(len(data)))
    random.shuffle(order)
    for i in range(0, len(order), BATCH):
        chunk = [data[j] for j in order[i:i + BATCH]]
        n = max(len(x) for x, _ in chunk)
        ids = torch.tensor([x + [pad_id] * (n - len(x)) for x, _ in chunk])
        lab = torch.tensor([y + [-100] * (n - len(y)) for _, y in chunk])
        att = torch.tensor([[1] * len(x) + [0] * (n - len(x)) for x, _ in chunk])
        yield ids.to(device), att.to(device), lab.to(device)

random.seed(0)
model.train()
step, t0 = 0, time.time()
for epoch in range(EPOCHS):
    for k, (ids, att, lab) in enumerate(batches()):
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
            loss = model(input_ids=ids, attention_mask=att, labels=lab).loss / ACCUM
        loss.backward()
        if (k + 1) % ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            step += 1
            if step % 10 == 0:
                print(f"epoch {epoch + 1} step {step}/{steps} loss {loss.item() * ACCUM:.4f} ({time.time() - t0:.0f}s)")

## ⑤ 학습 후 정확도와 저장

In [ ]:
after = evaluate(show=True)
print(f"[{MODEL_CHOICE}] 학습 전 {before * 100:.1f}% → 학습 후 {after * 100:.1f}%")

if cfg["method"] == "lora":
    model = model.merge_and_unload()    # LoRA 보조 가중치를 원래 가중치에 합쳐 보통 모델로 만든다
model.save_pretrained(OUT_DIR)

# 토크나이저는 학습하지 않았으므로 원본 파일을 그대로 복사한다.
# (transformers 5.x가 다시 저장한 토크나이저는 GGUF 변환기의 어휘 크기 검사에서 실패할 수 있다)
from huggingface_hub import hf_hub_download, list_repo_files
TOKENIZER_FILES = {"tokenizer.json", "tokenizer.model", "tokenizer_config.json", "special_tokens_map.json",
                   "added_tokens.json", "chat_template.jinja", "vocab.json", "merges.txt"}
for name in list_repo_files(MODEL_ID):
    if name in TOKENIZER_FILES:
        shutil.copy(hf_hub_download(MODEL_ID, name), os.path.join(OUT_DIR, name))
print(sorted(os.listdir(OUT_DIR)))

## ⑥ GGUF 변환

llama.cpp의 변환 스크립트로 Jetson용 GGUF(Q8_0)를 만든다. FunctionGemma 약 280MB, Qwen3 0.6B 약 610MB.

In [ ]:
if not os.path.exists("llama.cpp"):
    !git clone -q --depth 1 https://github.com/ggml-org/llama.cpp.git
!pip install -q gguf sentencepiece protobuf
!python llama.cpp/convert_hf_to_gguf.py {OUT_DIR} --outfile {OUT_DIR}-Q8_0.gguf --outtype q8_0
!ls -lh *.gguf

## ⑦ 다운로드 → Jetson에서 실행

1. 아래 셀로 GGUF 파일을 PC에 받는다.
2. PC(PowerShell)에서 Jetson으로 복사한다: `scp <파일이름>.gguf hansungai@<JETSON_IP>:~/models/`
3. Jetson에서 `./start_server.sh ft-functiongemma` 또는 `./start_server.sh ft-qwen3-0.6b`로 서버를 켜고 `fc_eval.py`로 다시 측정한다 (실습 문서 8장).

In [ ]:
from google.colab import files
files.download(f"{OUT_DIR}-Q8_0.gguf")